# # Multi-Level Fusion Hybrid Model for Chest X-Ray Multi-Label Classification
This notebook implements a sophisticated hybrid architecture that combines:
- **ResNet50** with **CBAM attention** at multiple levels
- **Vision Transformer (ViT)** for global context
- **Multi-level feature fusion** at different scales
- **Custom weighted focal loss** for imbalanced classes
- **Progressive training strategy**

In [ ]:
# 1. Setup and Imports
import numpy as np
import pandas as pd
import os
import glob
import tensorflow as tf
from keras.preprocessing.image import ImageDataGenerator
from keras.applications import ResNet50
from keras.layers import (Dense, GlobalAveragePooling2D, Input, Reshape, LayerNormalization, 
                         MultiHeadAttention, Dropout, Add, GlobalAveragePooling1D, 
                         BatchNormalization, Conv2D, Multiply, Lambda, Concatenate,
                         MaxPooling2D, AveragePooling2D, Activation, UpSampling2D)
from keras.models import Model
from keras.optimizers import Adam
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from keras import backend as K
from keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from sklearn.metrics import precision_recall_curve, auc, roc_curve, roc_auc_score
import tensorflow_addons as tfa
import warnings
warnings.filterwarnings('ignore')

# Enable mixed precision for GTX 1060 memory savings
tf.keras.mixed_precision.set_global_policy('mixed_float16')

In [ ]:
# 2. Data Loading
# Dataset paths
BASE_DIR = "./dataset_balanced/"
CSV_PATH = os.path.join(BASE_DIR, "new_labels.csv")
IMAGE_DIR = os.path.join(BASE_DIR, "new_images")
IMAGE_FILES = {os.path.basename(f): f for f in glob.glob(os.path.join(IMAGE_DIR, "*.png"))}

print(f"Total images found: {len(IMAGE_FILES)}")

# Disease labels
labels = ['Atelectasis', 'Cardiomegaly', 'Consolidation', 'Edema', 'Effusion',
          'Emphysema', 'Fibrosis', 'Hernia', 'Infiltration', 'Mass',
          'Nodule', 'Pleural_Thickening', 'Pneumonia', 'Pneumothorax']

In [ ]:
# Load CSV and analyze class distribution
df = pd.read_csv(CSV_PATH)
df = df[['Path'] + labels]

# Analyze class imbalance
class_counts = df[labels].sum()
class_weights = len(df) / (len(labels) * class_counts)  # Inverse frequency weighting

print("Class Distribution Analysis:")
print("="*50)
for label, count, weight in zip(labels, class_counts, class_weights):
    percentage = (count / len(df)) * 100
    print(f"{label:20s}: {count:5d} ({percentage:5.2f}%) - Weight: {weight:.3f}")

# Visualize class distribution
plt.figure(figsize=(15, 6))
plt.bar(range(len(labels)), class_counts, color='skyblue', alpha=0.7)
plt.xticks(range(len(labels)), labels, rotation=45, ha='right')
plt.ylabel('Number of Positive Cases')
plt.title('Class Distribution - Multi-Label Chest X-Ray Dataset')
plt.tight_layout()
plt.show()

In [ ]:
# Create stratification for train/val split
df['disease_count'] = df[labels].sum(axis=1)
df['has_any_disease'] = (df['disease_count'] > 0).astype(int)

# Stratified split
train_df, val_df = train_test_split(
    df, 
    test_size=0.2, 
    random_state=42,
    stratify=df['has_any_disease']
)

print(f"Train samples: {len(train_df)}")
print(f"Validation samples: {len(val_df)}")
print(f"Train positive cases: {train_df['has_any_disease'].sum()}")
print(f"Validation positive cases: {val_df['has_any_disease'].sum()}")

In [ ]:
# Augmentation
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=8,            # Subtle rotations
    width_shift_range=0.08,      # Small shifts
    height_shift_range=0.08,
    horizontal_flip=False,       # Never flip chest X-rays
    zoom_range=0.05,            # Minimal zoom
    brightness_range=[0.9, 1.1], # Slight brightness variation
    fill_mode='nearest',
    shear_range=0.02            # Very subtle shear
)

val_datagen = ImageDataGenerator(rescale=1./255)

# Add image paths
train_df['path'] = train_df['Path'].map(IMAGE_FILES)
val_df['path'] = val_df['Path'].map(IMAGE_FILES)

# Create generators with smaller batch size for GTX 1060
BATCH_SIZE = 4
train_gen = train_datagen.flow_from_dataframe(
    dataframe=train_df,
    directory=None,
    x_col='path',
    y_col=labels,
    target_size=(224, 224),
    batch_size=BATCH_SIZE,
    class_mode='raw',
    shuffle=True
)

val_gen = val_datagen.flow_from_dataframe(
    dataframe=val_df,
    directory=None,
    x_col='path',
    y_col=labels,
    target_size=(224, 224),
    batch_size=BATCH_SIZE,
    class_mode='raw',
    shuffle=False
)

In [ ]:
# 4. Custom Loss Function for Imbalanced Multi-Label Classification

class WeightedAsymmetricFocalLoss(tf.keras.losses.Loss):
    """
    Custom loss combining:
    1. Weighted loss for class imbalance
    2. Asymmetric loss for multi-label
    3. Focal loss for hard examples
    """
    
    def __init__(self, class_weights, gamma_pos=1.0, gamma_neg=4.0, 
                 clip=0.05, eps=1e-8, name='weighted_asymmetric_focal_loss'):
        super().__init__(name=name)
        self.class_weights = tf.constant(class_weights, dtype=tf.float32)
        self.gamma_pos = gamma_pos
        self.gamma_neg = gamma_neg
        self.clip = clip
        self.eps = eps
    
    def call(self, y_true, y_pred):
        # Clip predictions to prevent log(0)
        y_pred = tf.clip_by_value(y_pred, self.eps, 1 - self.eps)
        
        # Asymmetric focusing
        if self.gamma_neg > 0 or self.gamma_pos > 0:
            if self.clip > 0:
                pt = y_pred * y_true + (1 - y_pred) * (1 - y_true)
                pt = tf.clip_by_value(pt, self.clip, 1)
            else:
                pt = y_pred * y_true + (1 - y_pred) * (1 - y_true)
            
            # Asymmetric gamma
            gamma = self.gamma_pos * y_true + self.gamma_neg * (1 - y_true)
            focal_weight = (1 - pt) ** gamma
        else:
            focal_weight = 1.0
        
        # Binary cross entropy
        bce = -(y_true * tf.math.log(y_pred) + (1 - y_true) * tf.math.log(1 - y_pred))
        
        # Apply focal weight
        focal_loss = focal_weight * bce
        
        # Apply class weights (broadcast across batch dimension)
        weighted_loss = focal_loss * self.class_weights
        
        return tf.reduce_mean(tf.reduce_sum(weighted_loss, axis=1))

# Create custom loss with computed class weights
custom_loss = WeightedAsymmetricFocalLoss(
    class_weights=class_weights.values,
    gamma_pos=0.5,  # Less focusing on easy positives
    gamma_neg=3.0,  # More focusing on hard negatives
    clip=0.05
)
print(f"Class weights range: {class_weights.min():.3f} - {class_weights.max():.3f}")

In [ ]:
# 5. CBAM Attention Modules

def channel_attention(input_feature, ratio=16, name_prefix=""):
    """Enhanced Channel Attention with different pooling strategies"""
    channel = input_feature.shape[-1]
    
    # Shared MLP layers
    shared_layer_one = Dense(channel // ratio, activation='relu', 
                           name=f'{name_prefix}_ca_dense1')
    shared_layer_two = Dense(channel, name=f'{name_prefix}_ca_dense2')
    
    # Multiple pooling strategies
    avg_pool = GlobalAveragePooling2D(name=f'{name_prefix}_ca_avg_pool')(input_feature)
    max_pool = tf.keras.layers.GlobalMaxPooling2D(name=f'{name_prefix}_ca_max_pool')(input_feature)
    
    avg_pool = Reshape((1, 1, channel))(avg_pool)
    max_pool = Reshape((1, 1, channel))(max_pool)
    
    # Process through shared MLP
    avg_pool = shared_layer_one(avg_pool)
    avg_pool = shared_layer_two(avg_pool)
    
    max_pool = shared_layer_one(max_pool)
    max_pool = shared_layer_two(max_pool)
    
    # Combine and apply sigmoid
    channel_attention_feature = Add(name=f'{name_prefix}_ca_add')([avg_pool, max_pool])
    channel_attention_feature = Activation('sigmoid', name=f'{name_prefix}_ca_sigmoid')(channel_attention_feature)
    
    return Multiply(name=f'{name_prefix}_ca_multiply')([input_feature, channel_attention_feature])

def spatial_attention(input_feature, kernel_size=7, name_prefix=""):
    """Enhanced Spatial Attention"""
    # Channel-wise pooling
    avg_pool = Lambda(lambda x: K.mean(x, axis=3, keepdims=True), 
                     name=f'{name_prefix}_sa_avg_pool')(input_feature)
    max_pool = Lambda(lambda x: K.max(x, axis=3, keepdims=True),
                     name=f'{name_prefix}_sa_max_pool')(input_feature)
    
    # Concatenate
    concat = Concatenate(axis=3, name=f'{name_prefix}_sa_concat')([avg_pool, max_pool])
    
    # Spatial attention convolution
    spatial_attention_feature = Conv2D(filters=1, kernel_size=kernel_size, 
                                     strides=1, padding='same', activation='sigmoid',
                                     name=f'{name_prefix}_sa_conv')(concat)
    
    return Multiply(name=f'{name_prefix}_sa_multiply')([input_feature, spatial_attention_feature])

def cbam_block(input_feature, ratio=16, kernel_size=7, name_prefix="cbam"):
    """Complete CBAM block with proper naming"""
    cbam_feature = channel_attention(input_feature, ratio, f"{name_prefix}_channel")
    cbam_feature = spatial_attention(cbam_feature, kernel_size, f"{name_prefix}_spatial")
    return cbam_feature

In [ ]:
# 6. Vision Transformer Components

class PatchEmbedding(tf.keras.layers.Layer):
    """Efficient patch embedding layer"""
    def __init__(self, patch_size=16, embed_dim=384, **kwargs):
        super().__init__(**kwargs)
        self.patch_size = patch_size
        self.embed_dim = embed_dim
        self.projection = Conv2D(embed_dim, kernel_size=patch_size, 
                               strides=patch_size, padding='valid')
        
    def call(self, images):
        batch_size = tf.shape(images)[0]
        patches = self.projection(images)
        h, w = patches.shape[1], patches.shape[2]
        patches = Reshape((h * w, self.embed_dim))(patches)
        return patches, h * w

class MultiHeadSelfAttention(tf.keras.layers.Layer):
    """Memory-efficient multi-head self-attention"""
    def __init__(self, embed_dim, num_heads=8, **kwargs):
        super().__init__(**kwargs)
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        
        self.qkv = Dense(embed_dim * 3, use_bias=False)
        self.proj = Dense(embed_dim)
        self.dropout = Dropout(0.1)
        
    def call(self, x, training=None):
        batch_size, seq_len, embed_dim = tf.shape(x)[0], tf.shape(x)[1], x.shape[2]
        
        # Generate Q, K, V
        qkv = self.qkv(x)
        qkv = Reshape((seq_len, 3, self.num_heads, self.head_dim))(qkv)
        qkv = tf.transpose(qkv, perm=[2, 0, 3, 1, 4])
        q, k, v = qkv[0], qkv[1], qkv[2]
        
        # Scaled dot-product attention
        scale = tf.cast(self.head_dim, dtype=tf.float32) ** -0.5
        attn = tf.matmul(q, k, transpose_b=True) * scale
        attn = tf.nn.softmax(attn, axis=-1)
        attn = self.dropout(attn, training=training)
        
        # Apply attention to values
        out = tf.matmul(attn, v)
        out = tf.transpose(out, perm=[0, 2, 1, 3])
        out = Reshape((seq_len, embed_dim))(out)
        out = self.proj(out)
        
        return out

class TransformerBlock(tf.keras.layers.Layer):
    """Transformer encoder block"""
    def __init__(self, embed_dim, num_heads, ff_dim, dropout_rate=0.1, **kwargs):
        super().__init__(**kwargs)
        self.att = MultiHeadSelfAttention(embed_dim, num_heads)
        self.ffn = tf.keras.Sequential([
            Dense(ff_dim, activation='gelu'),
            Dense(embed_dim)
        ])
        self.layernorm1 = LayerNormalization(epsilon=1e-6)
        self.layernorm2 = LayerNormalization(epsilon=1e-6)
        self.dropout1 = Dropout(dropout_rate)
        self.dropout2 = Dropout(dropout_rate)
    
    def call(self, inputs, training=None):
        # Self-attention
        attn_output = self.att(inputs, training=training)
        attn_output = self.dropout1(attn_output, training=training)
        out1 = self.layernorm1(inputs + attn_output)
        
        # Feed-forward
        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output, training=training)
        out2 = self.layernorm2(out1 + ffn_output)
        
        return out2

In [ ]:
# 7. Multi-Level Feature Fusion Architecture

def multi_level_fusion_model(
    input_shape=(224, 224, 3),
    num_classes=14,
    # CNN parameters
    backbone_trainable_layers=20,
    cbam_ratios=[16, 8, 4],  # Different ratios for multi-level
    # ViT parameters  
    patch_size=16,
    embed_dim=384,
    num_transformer_blocks=3,
    num_heads=6,
    ff_dim=768,
    transformer_dropout=0.1,
    # Fusion parameters
    fusion_levels=['low', 'mid', 'high'],
    fusion_dims=[128, 256, 512],
    final_dim=768
):
    """
    Multi-level fusion architecture:
    1. Extract features at multiple CNN levels
    2. Apply CBAM at each level
    3. Combine with ViT global features
    4. Hierarchical fusion strategy
    """
    
    inputs = Input(shape=input_shape, name='input_images')
    
    
    

In [ ]:
# =============== CNN BACKBONE WITH MULTI-LEVEL EXTRACTION ===============
    print("Building multi-level CNN backbone...")
    
    # ResNet50 backbone
    resnet_base = ResNet50(weights='imagenet', include_top=False, input_shape=input_shape)
    
    # Freeze early layers
    for i, layer in enumerate(resnet_base.layers):
        if i < len(resnet_base.layers) - backbone_trainable_layers:
            layer.trainable = False
        else:
            layer.trainable = True
    
    # Extract intermediate features at different levels
    # Low-level: after conv2_block3 (56x56)
    low_level_layer = resnet_base.get_layer('conv2_block3_out')
    # Mid-level: after conv3_block4 (28x28)  
    mid_level_layer = resnet_base.get_layer('conv3_block4_out')
    # High-level: after conv4_block6 (14x14)
    high_level_layer = resnet_base.get_layer('conv4_block6_out')
    # Top-level: final output (7x7)
    top_level_output = resnet_base(inputs)
    
    # Extract multi-level features
    low_features = low_level_layer.output      # (None, 56, 56, 256)
    mid_features = mid_level_layer.output      # (None, 28, 28, 512)  
    high_features = high_level_layer.output    # (None, 14, 14, 1024)
    top_features = top_level_output            # (None, 7, 7, 2048)
    
    # Apply CBAM at each level with different ratios
    low_cbam = cbam_block(low_features, ratio=cbam_ratios[0], name_prefix="low_level")
    mid_cbam = cbam_block(mid_features, ratio=cbam_ratios[1], name_prefix="mid_level")
    high_cbam = cbam_block(high_features, ratio=cbam_ratios[2], name_prefix="high_level")
    top_cbam = cbam_block(top_features, ratio=8, name_prefix="top_level")

In [ ]:
# =============== VISION TRANSFORMER BRANCH ===============
    print("Building Vision Transformer branch...")
    
    # Patch embedding
    patch_embed = PatchEmbedding(patch_size=patch_size, embed_dim=embed_dim, name='patch_embedding')
    patches, num_patches = patch_embed(inputs)
    
    # Positional embedding
    pos_embed = tf.keras.layers.Embedding(num_patches, embed_dim, name='pos_embedding')
    positions = tf.range(start=0, limit=num_patches, delta=1)
    patches = Add(name='add_pos_embed')([patches, pos_embed(positions)])
    
    # Transformer blocks
    vit_features = patches
    for i in range(num_transformer_blocks):
        vit_features = TransformerBlock(
            embed_dim=embed_dim,
            num_heads=num_heads, 
            ff_dim=ff_dim,
            dropout_rate=transformer_dropout,
            name=f'transformer_block_{i}'
        )(vit_features)
    
    # Global average pooling for ViT
    vit_global = GlobalAveragePooling1D(name='vit_global_pool')(vit_features)

In [ ]:
# =============== MULTI-LEVEL FEATURE FUSION ===============
    print("Building multi-level fusion...")
    
    # Adaptive pooling to standardize spatial dimensions
    def adaptive_pool_and_project(features, target_dim, level_name):
        # Global average pooling
        pooled = GlobalAveragePooling2D(name=f'{level_name}_gap')(features)
        # Project to target dimension
        projected = Dense(target_dim, activation='relu', name=f'{level_name}_proj')(pooled)
        projected = BatchNormalization(name=f'{level_name}_bn')(projected)
        projected = Dropout(0.2, name=f'{level_name}_dropout')(projected)
        return projected
    
    # Process each level
    low_processed = adaptive_pool_and_project(low_cbam, fusion_dims[0], 'low')
    mid_processed = adaptive_pool_and_project(mid_cbam, fusion_dims[1], 'mid') 
    high_processed = adaptive_pool_and_project(high_cbam, fusion_dims[2], 'high')
    top_processed = adaptive_pool_and_project(top_cbam, fusion_dims[2], 'top')
    
    # Process ViT features
    vit_processed = Dense(fusion_dims[2], activation='relu', name='vit_proj')(vit_global)
    vit_processed = BatchNormalization(name='vit_bn')(vit_processed)
    vit_processed = Dropout(0.2, name='vit_dropout')(vit_processed)
    
    # Hierarchical fusion strategy
    # Level 1: Combine low and mid level features
    level1_concat = Concatenate(name='level1_concat')([low_processed, mid_processed])
    level1_fused = Dense(fusion_dims[1], activation='relu', name='level1_fusion')(level1_concat)
    level1_fused = BatchNormalization(name='level1_bn')(level1_fused)
    
    # Level 2: Combine level1 with high-level features  
    level2_concat = Concatenate(name='level2_concat')([level1_fused, high_processed])
    level2_fused = Dense(fusion_dims[2], activation='relu', name='level2_fusion')(level2_concat)
    level2_fused = BatchNormalization(name='level2_bn')(level2_fused)
    
    # Level 3: Combine with top CNN features
    level3_concat = Concatenate(name='level3_concat')([level2_fused, top_processed])
    level3_fused = Dense(fusion_dims[2], activation='relu', name='level3_fusion')(level3_concat)
    level3_fused = BatchNormalization(name='level3_bn')(level3_fused)
    
    # Final fusion: CNN hierarchy + ViT global
    final_concat = Concatenate(name='final_concat')([level3_fused, vit_processed])
    
    # Cross-attention between CNN and ViT features
    cnn_query = Dense(256, name='cnn_query')(level3_fused)
    vit_key = Dense(256, name='vit_key')(vit_processed)
    vit_value = Dense(256, name='vit_value')(vit_processed)
    
    # Simple attention mechanism
    attention_scores = tf.keras.layers.Dot(axes=1, name='attention_scores')([cnn_query, vit_key])
    attention_weights = Activation('softmax', name='attention_weights')(attention_scores)
    attended_vit = Multiply(name='attended_vit')([attention_weights, vit_value])
    
    # Final feature combination
    final_features = Concatenate(name='final_features')([final_concat, attended_vit])
    final_features = Dense(final_dim, activation='relu', name='final_projection')(final_features)
    final_features = BatchNormalization(name='final_bn')(final_features)
    final_features = Dropout(0.4, name='final_dropout')(final_features)
    

In [ ]:
# =============== CLASSIFICATION HEAD ===============
    # Multi-scale classification
    x = final_features
    
    # First classification layer
    x = Dense(512, activation='relu', name='cls_dense1')(x)
    x = BatchNormalization(name='cls_bn1')(x)
    x = Dropout(0.3, name='cls_dropout1')(x)
    
    # Second classification layer  
    x = Dense(256, activation='relu', name='cls_dense2')(x)
    x = BatchNormalization(name='cls_bn2')(x)
    x = Dropout(0.3, name='cls_dropout2')(x)
    
    # Final output layer (float32 for mixed precision)
    outputs = Dense(num_classes, activation='sigmoid', dtype='float32', 
                   name='predictions')(x)
    
    # Create model
    model = Model(inputs=inputs, outputs=outputs, name='MultiLevel_Fusion_Model')
    
    return model, resnet_base

In [ ]:
# 8. Model Creation

# Create the multi-level fusion model
print("Creating multi-level fusion model...")
model, resnet_backbone = multi_level_fusion_model(
    input_shape=(224, 224, 3),
    num_classes=len(labels),
    backbone_trainable_layers=25,
    cbam_ratios=[16, 8, 4],
    patch_size=16,
    embed_dim=384,
    num_transformer_blocks=3,
    num_heads=6,
    ff_dim=768,
    transformer_dropout=0.1,
    fusion_levels=['low', 'mid', 'high'],
    fusion_dims=[128, 256, 512],
    final_dim=768
)
print(f"📊 Total parameters: {model.count_params():,}")

In [ ]:
model.summary(line_length=120)

In [ ]:
# 9. Progressive Training Strategy

def progressive_training_strategy():
    """
    Multi-stage progressive training:
    1. Warm-up: Train fusion layers and classifier only
    2. Fine-tuning: Unfreeze some CNN layers with custom loss
    3. Final: Full fine-tuning with very low learning rate
    """
    
    # ============= STAGE 1: WARM-UP TRAINING =============
    # STAGE 1: Warm-up Training (Frozen Backbone)")
    
    # Freeze all backbone layers
    for layer in resnet_backbone.layers:
        layer.trainable = False
    
    # Compile with standard BCE for warm-up
    model.compile(
        optimizer=Adam(learning_rate=2e-3, beta_1=0.9, beta_2=0.999, epsilon=1e-7),
        loss='binary_crossentropy',
        metrics=[
            'binary_accuracy',
            tf.keras.metrics.AUC(name='auc'),
            tf.keras.metrics.Precision(name='precision'),
            tf.keras.metrics.Recall(name='recall')
        ]
    )
    
    # Stage 1 callbacks
    stage1_callbacks = [
        EarlyStopping(monitor='val_auc', patience=5, mode='max', 
                     restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, 
                         min_lr=1e-6, verbose=1)
    ]
    
    # Training fusion layers and classifier...
    history1 = model.fit(
        train_gen,
        validation_data=val_gen,
        epochs=12,
        callbacks=stage1_callbacks,
        verbose=1
    )

In [ ]:
# ============= STAGE 2: CUSTOM LOSS FINE-TUNING =============
# STAGE 2: Custom Loss Fine-tuning
    
    # Unfreeze top CNN layers gradually
    trainable_layers = 30
    for layer in resnet_backbone.layers[-trainable_layers:]:
        layer.trainable = True
    
    print(f"Unfrozen {trainable_layers} top ResNet layers")
    
    # Compile with custom weighted loss
    model.compile(
        optimizer=Adam(learning_rate=5e-5, beta_1=0.9, beta_2=0.999, epsilon=1e-7),
        loss=custom_loss,
        metrics=[
            'binary_accuracy',
            tf.keras.metrics.AUC(name='auc'),
            tf.keras.metrics.Precision(name='precision'),
            tf.keras.metrics.Recall(name='recall')
        ]
    )
    
    # Stage 2 callbacks
    stage2_callbacks = [
        EarlyStopping(monitor='val_auc', patience=7, mode='max', 
                     restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor='val_auc', factor=0.3, patience=4, 
                         min_lr=1e-7, mode='max', verbose=1),
        ModelCheckpoint('./output/multi_level_fusion_stage2.keras', 
                       save_best_only=True, monitor='val_auc', mode='max', verbose=1)
    ]
    
    # Fine-tuning with custom weighted asymmetric focal loss...
    history2 = model.fit(
        train_gen,
        validation_data=val_gen,
        epochs=18,
        callbacks=stage2_callbacks,
        verbose=1
    )